<a href="https://colab.research.google.com/github/joseportocarrero-stack/DataScience-Homework/blob/main/FGD_C28R_1A_LABD11_Portocarrero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATORIO DIRIGIDO N.° 11

## Gestión de Metadatos: el catálogo de datos que gobierna el sistema

**Curso:** Fundamentos de Gestión de Datos · TECSUP
**Docente:** Pilar Rocío Sayán Mejía
**Caso:** Farmacia MediSur — base oficial del PMD2
**Duración:** 1 hora y 40 minutos

---

### Elemento de la capacidad terminal

Construye el catálogo de metadatos de la base del proyecto (técnicos, de negocio, de sensibilidad y de linaje), lo almacena como una tabla dentro de la propia base y lo utiliza para controlar, mediante roles de acceso, qué información puede ver cada usuario del sistema a través de un menú interactivo.

## Caso introductorio

En la Semana 10 migraste la base de **Farmacia MediSur** y dejaste una base limpia (`farmacia_migrada.db`). Hoy llega un problema nuevo.

MediSur contrató personal: vendedores en mostrador, cajeros, un supervisor de sede y un auditor externo. Todos necesitan consultar la base, pero **no todos pueden ver lo mismo**. La tabla `clientes` guarda la columna `condicion_cronica`: si un cliente es diabético o hipertenso. Ese dato es información de salud, y un vendedor de mostrador no tiene por qué verlo.

El gerente te hace dos preguntas que hoy **no puedes responder**:

1. *"¿Qué significa exactamente cada columna de la base y de dónde salió?"* — Nadie lo documentó. Solo tú, que hiciste el ETL, sabes que `correo_fue_corregido` existe porque tu pipeline anuló los correos inválidos. Si mañana renuncias, ese conocimiento se va contigo.
2. *"¿Quién puede ver los datos de salud de mis clientes?"* — Hoy, cualquiera que abra el notebook.

Ambas preguntas se resuelven con lo mismo: un **catálogo de metadatos**. Metadatos son *datos sobre los datos*: no el valor `"Diabetes"`, sino el hecho de que la columna `condicion_cronica` contiene información de salud, es sensible, la ingresa el personal médico y solo el auditor puede consultarla.

Y aquí está la clave de este laboratorio: el catálogo **no es un documento que nadie lee**. Es una tabla dentro de la base, y el sistema la consulta para decidir qué columnas mostrarle a cada rol. Los metadatos gobiernan el menú.

Al terminar hoy tendrás la primera versión ejecutable del sistema del PMD2: eliges tu rol, entras a un menú y navegas por el catálogo de productos, los clientes y el catálogo de datos. En las semanas 12 a 15 ese mismo menú irá creciendo.

## Actividad 1: conceptos previos

Antes de programar, completa el cuadro con tus propias palabras. No copies definiciones de internet: escribe lo que entendiste.

| # | Concepto | Definición con tus propias palabras |
|---|---|---|
| 1 | Metadato | Información que ayuda a entender y usar un dato.|
| 2 | Metadato técnico | Datos que describen cómo está construido un dato.|
| 3 | Metadato de negocio | Datos que describen el significado y uso de un dato por un negocio.|
| 4 | Metadato operacional | Datos de describen la logística de un dato.|
| 5 | Catálogo de datos | Conjunto completo y estandarizado de metadatos, que incluye la trazabilidad.|
| 6 | Linaje o trazabilidad del dato | Documentación o registro de cada uno de los cambios o procesos por los que pasa un dato desde su fuente de origen.|
| 7 | Dato sensible | Datos que permiten la identificación de una persona, y que tienen la mayor protección legal.|
| 8 | Rol de acceso | Se refiere a los usuarios de una base de datos, y la autorización de la que disponen para acceder, e incluso modificar los datos de una organización.|
| 9 | Dublin Core | Estándar básico y ampliamente difundido de catalogación de metadatos.|
| 10 | ISO 11179 | Estándar muy completo para la catalogación de metadatos, a nivel de alta organización o especialización. Facilita la compatibilidad entre sistemas.|

### Los tres tipos de metadatos

Toda columna de una base se puede describir desde tres ángulos distintos. Usemos `condicion_cronica` de MediSur como ejemplo:

| Tipo | Qué responde | Ejemplo en `condicion_cronica` | ¿Se puede automatizar? |
|---|---|---|---|
| **Técnico** | ¿Cómo está guardado? | Tipo `TEXT`, admite nulos, no es clave primaria | **Sí**, el motor lo sabe |
| **De negocio** | ¿Qué significa para la empresa? | Enfermedad crónica declarada por el cliente; sirve para alertas de interacción entre medicamentos | **No**, lo redacta quien conoce el negocio |
| **Operacional** | ¿De dónde vino y cuándo? | Viene de `farmacia_legacy.db`, pasó por el ETL de la S10, se actualiza en cada compra | **Parcialmente** |

Esta distinción es el corazón de la semana: **el motor de base de datos te regala los metadatos técnicos, pero los de negocio los tiene que escribir una persona.** Por eso los catálogos de datos fracasan en las empresas: nadie quiere escribir esa columna.

### Los niveles de sensibilidad

Vamos a clasificar cada columna en uno de cuatro niveles. El nivel decide quién la ve:

| Nivel | Qué es | Ejemplo en MediSur |
|---|---|---|
| **Público** | Cualquiera puede verlo, incluso fuera de la empresa | `nombre_producto`, `precio_base` |
| **Interno** | Personal de la empresa | `nombres`, `apellidos`, `distrito`, `segmento` |
| **Confidencial** | Dato personal identificable (PII) | `num_documento`, `correo`, `telefono` |
| **Sensible** | Categoría especial protegida por ley | `condicion_cronica` (dato de salud) |

En el Perú, la **Ley 29733 de Protección de Datos Personales** trata los datos de salud como *dato sensible*, con protección reforzada. En Europa, el **GDPR** los llama *categoría especial*. Ambas normas coinciden: los datos de salud no se muestran a cualquiera.

Fíjate en un detalle que parece un error y no lo es: `nombres` **es** un dato personal, pero está en Interno, no en Confidencial. Es una decisión deliberada. Un vendedor no puede atender a un cliente sin saber a quién atiende. **Clasificar de más es tan dañino como clasificar de menos**: si el vendedor no puede hacer su trabajo, va a terminar pidiéndole la clave al Data Owner, y ahí se acabó todo el control de acceso. La seguridad que estorba se termina burlando.

## Actividad 2: desarrollo práctico en Colab

---

### Paso 1: preparar la base del proyecto

Este laboratorio continúa el de la Semana 10, así que trabajamos sobre **tu base migrada** (`farmacia_migrada.db`).

Si la perdiste o cerraste Colab, no hay problema: la celda la reconstruye sola descargando la base oficial y volviendo a ejecutar el ETL de la S10 en versión resumida.

Además hacemos algo nuevo. La migración de la S10 se centró en `clientes` y `operaciones`, pero el sistema necesita también los productos, los pagos y el detalle de cada venta. Vamos a **completar el destino** con las tablas que el ETL no tocó. Esto es exactamente lo que ocurre en una migración real: se hace por fases, primero lo crítico y después el resto.

In [ ]:
# ============================================================
# Paso 1: preparar la base del proyecto (continuidad con la S10)
# ============================================================
import sqlite3
import os
import pandas as pd
import numpy as np

caso = "03_farmacia_medisur"          # <-- cambia por la carpeta de TU caso asignado
DB_ORIGEN  = "farmacia_legacy.db"     # base oficial, sucia (la de siempre)
DB_DESTINO = "farmacia_migrada.db"    # base del proyecto (la que migraste en la S10)

# Repositorio oficial de las bases del curso.
REPO = "Rociosayan/-PMD2_FGD_Bases_Oficiales_FDG"
url  = f"https://raw.githubusercontent.com/{REPO}/main/casos/{caso}/{caso}.db"


def descargar_base_oficial():
    """Trae la base oficial del caso desde el repositorio del curso."""
    if os.path.exists(DB_ORIGEN):
        print("La base oficial ya estaba descargada.")
        return
    import requests
    r = requests.get(url)
    if r.status_code == 200 and r.content[:16] == b"SQLite format 3\x00":
        with open(DB_ORIGEN, "wb") as f:
            f.write(r.content)
        print("Base oficial descargada:", len(r.content), "bytes")
    else:
        from google.colab import files
        print("No se pudo descargar. Sube manualmente el .db de tu caso:")
        subida = files.upload()
        with open(DB_ORIGEN, "wb") as f:
            f.write(list(subida.values())[0])


def rehacer_etl_semana10():
    """Version resumida del pipeline de la S10, por si perdiste farmacia_migrada.db."""
    o = sqlite3.connect(DB_ORIGEN)
    d = sqlite3.connect(DB_DESTINO)

    # --- clientes: quitar DNI duplicados, estandarizar texto, anular correos invalidos
    cli = pd.read_sql_query("SELECT * FROM clientes", o)
    cli = cli.drop_duplicates(subset=["num_documento"], keep="first")
    for col in ["nombres", "apellidos", "distrito", "segmento", "tipo_documento"]:
        cli[col] = cli[col].astype("string").str.strip().str.title()
    formato_ok = cli["correo"].str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)
    cli["correo_fue_corregido"] = cli["correo"].notna() & ~formato_ok
    cli.loc[~formato_ok, "correo"] = pd.NA
    cli["correo"] = cli["correo"].fillna("SIN CORREO REGISTRADO")
    cli["telefono"] = cli["telefono"].fillna("000000000")
    cli.to_sql("clientes_limpio", d, if_exists="replace", index=False)

    # --- ventas: corregir montos negativos y marcar huerfanas
    op = pd.read_sql_query("SELECT * FROM operaciones", o)
    ids_cli = set(pd.read_sql_query("SELECT id_cliente FROM clientes", o)["id_cliente"])
    ids_emp = set(pd.read_sql_query("SELECT id_empleado FROM empleados", o)["id_empleado"])
    op["monto_original"]   = op["monto_total"]
    op["monto_ajustado"]   = np.where(op["monto_total"] < 0, 0, op["monto_total"])
    op["monto_fue_ajustado"] = op["monto_total"] < 0
    op["cliente_valido"]   = op["id_cliente"].isin(ids_cli)
    op["empleado_valido"]  = op["id_empleado"].isin(ids_emp)
    op.to_sql("ventas_limpia", d, if_exists="replace", index=False)

    d.commit(); o.close(); d.close()
    print("ETL de la S10 reconstruido: clientes_limpio y ventas_limpia.")


def completar_tablas_de_apoyo():
    """Copia al destino las tablas que el ETL de la S10 no migro (fase 2 de la migracion)."""
    o = sqlite3.connect(DB_ORIGEN)
    d = sqlite3.connect(DB_DESTINO)
    ya_estan = set(pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'", d)["name"])

    pendientes = ["sedes", "empleados", "productos_servicios",
                  "detalle_operacion", "pagos", "incidencias"]
    copiadas = []
    for t in pendientes:
        if t in ya_estan:
            continue
        df = pd.read_sql_query(f"SELECT * FROM {t}", o)
        df.to_sql(t, d, if_exists="replace", index=False)
        copiadas.append(f"{t} ({len(df)})")

    d.commit(); o.close(); d.close()
    print("Tablas de apoyo copiadas:", ", ".join(copiadas) if copiadas else "ninguna (ya estaban)")


# --- ejecucion
descargar_base_oficial()
if not os.path.exists(DB_DESTINO):
    print("No encontre farmacia_migrada.db -> reconstruyo el ETL de la Semana 10.")
    rehacer_etl_semana10()
else:
    print("Base migrada de la S10 encontrada. Continuamos sobre ella.")
completar_tablas_de_apoyo()

# --- verificacion
conn = sqlite3.connect(DB_DESTINO)
tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print("\nLa base del proyecto tiene", len(tablas), "tablas:")
for t in tablas["name"]:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0]
    print(f"   - {t:22s} {n:5d} registros")

Base oficial descargada: 344064 bytes
No encontre farmacia_migrada.db -> reconstruyo el ETL de la Semana 10.
ETL de la S10 reconstruido: clientes_limpio y ventas_limpia.
Tablas de apoyo copiadas: sedes (6), empleados (40), productos_servicios (30), detalle_operacion (2572), pagos (1000), incidencias (155)

La base del proyecto tiene 8 tablas:
   - clientes_limpio          202 registros
   - detalle_operacion       2572 registros
   - empleados                 40 registros
   - incidencias              155 registros
   - pagos                   1000 registros
   - productos_servicios       30 registros
   - sedes                      6 registros
   - ventas_limpia           1010 registros


### Paso 2: metadatos TÉCNICOS (el motor los regala)

Los metadatos técnicos ya existen: la base los guarda para poder funcionar. Solo hay que pedírselos.

`PRAGMA table_info(tabla)` le pregunta a SQLite la estructura de una tabla y devuelve, por cada columna: su nombre, su tipo de dato, si admite nulos y si es clave primaria. Recorremos todas las tablas y armamos un inventario completo.

Fíjate en el ahorro: en menos de veinte líneas documentamos toda la base. Esto es lo que **no** hay que escribir a mano.

In [ ]:
# ============================================================
# Paso 2: extraer metadatos TECNICOS de toda la base
# ============================================================
# El catalogo describe los DATOS del negocio, no se describe a si mismo.
# Si no las excluimos, al ejecutar el notebook por segunda vez el catalogo
# encontraria su propia tabla y se cataria a si mismo: metadatos de metadatos.
TABLAS_DEL_SISTEMA = {"catalogo_metadatos", "ficha_dublin_core", "resumen_etl"}


def extraer_metadatos_tecnicos(conn):
    """Recorre las tablas de negocio y pregunta al motor la estructura de cada columna."""
    tablas = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)["name"]
    tablas = [t for t in tablas if t not in TABLAS_DEL_SISTEMA]

    filas = []
    for tabla in tablas:
        info = pd.read_sql_query(f"PRAGMA table_info({tabla})", conn)
        n_reg = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {tabla}", conn)["n"][0]
        for _, col in info.iterrows():
            filas.append({
                "tabla":        tabla,
                "columna":      col["name"],
                "tipo_dato":    col["type"] if col["type"] else "SIN TIPO",
                "admite_nulos": "No" if col["notnull"] == 1 else "Si",
                "es_clave_primaria": "Si" if col["pk"] == 1 else "No",
                "registros_tabla":   n_reg,
            })
    return pd.DataFrame(filas)


metadatos_tecnicos = extraer_metadatos_tecnicos(conn)

print("Columnas documentadas automaticamente:", len(metadatos_tecnicos))
print("Tablas cubiertas:", metadatos_tecnicos["tabla"].nunique())
print("\nEjemplo - estructura de clientes_limpio:")
display(metadatos_tecnicos[metadatos_tecnicos["tabla"] == "clientes_limpio"])

Columnas documentadas automaticamente: 68
Tablas cubiertas: 8

Ejemplo - estructura de clientes_limpio:


,tabla,columna,tipo_dato,admite_nulos,es_clave_primaria,registros_tabla
0,clientes_limpio,id_cliente,INTEGER,Si,No,202
1,clientes_limpio,codigo_cliente,TEXT,Si,No,202
2,clientes_limpio,tipo_documento,TEXT,Si,No,202
3,clientes_limpio,num_documento,TEXT,Si,No,202
4,clientes_limpio,nombres,TEXT,Si,No,202
5,clientes_limpio,apellidos,TEXT,Si,No,202
6,clientes_limpio,correo,TEXT,Si,No,202
7,clientes_limpio,telefono,TEXT,Si,No,202
8,clientes_limpio,distrito,TEXT,Si,No,202
9,clientes_limpio,segmento,TEXT,Si,No,202


**Pregunta 1:** ¿Cuántas columnas documentó el código automáticamente y cuánto habrías tardado en escribirlas a mano? Ahora mira la columna `tipo_dato`: ¿por qué crees que varias columnas aparecen con tipo `TEXT` aunque guarden fechas o números?

Respuesta: Se documentaron de forma automática 68 columnas, distribuidas en 8 tablas, con un número de registros variable entre 6 y 2572 por tabla, lo que da un total de 327 284 celdas o campos de valor. Si tardo unos 10,6 segundos en llenar un dato largo como un número de DNI, tardaría aproximadamente 963,67 horas o 40,15 días en escribir o digitar todos esos datos a mano. Si las columnas contienen números o fechas como datos de tipo cadena de texto, es porque no se va a efectuar ningún cálculo ni usar función de agregación alguna con estos datos.

### Paso 3: metadatos de NEGOCIO (los escribe una persona)

Aquí se acaba la automatización. Ningún programa sabe que `segmento` significa *categoría comercial del cliente según su frecuencia de compra*. Eso lo sabe quien conoce el negocio.

Por eso los catálogos de datos fracasan: la parte que aporta valor es la única que hay que escribir a mano, y nadie quiere hacerlo.

Fíjate en la columna `regla_negocio`: no describe el dato, describe **la regla que lo gobierna**. Es lo que evita que el próximo analista invente su propia interpretación.

In [ ]:
# ============================================================
# Paso 3: metadatos de NEGOCIO (redactados por quien conoce el negocio)
# ============================================================
# Formato:  (tabla, columna): (descripcion de negocio, regla de negocio)
NEGOCIO = {
    ("clientes_limpio", "id_cliente"):        ("Identificador interno del cliente en el sistema.", "Unico, no se reutiliza aunque el cliente se retire."),
    ("clientes_limpio", "codigo_cliente"):    ("Codigo comercial visible del cliente (C00001).", "Formato C + 5 digitos."),
    ("clientes_limpio", "tipo_documento"):    ("Tipo de documento de identidad presentado.", "DNI, CE o Pasaporte."),
    ("clientes_limpio", "num_documento"):     ("Numero de documento de identidad.", "DNI: 8 digitos. Es el identificador legal del cliente."),
    ("clientes_limpio", "nombres"):           ("Nombres del cliente.", "Solo letras. Se guarda con la primera en mayuscula."),
    ("clientes_limpio", "apellidos"):         ("Apellidos del cliente.", "Solo letras. Se guarda con la primera en mayuscula."),
    ("clientes_limpio", "correo"):            ("Correo de contacto para boletas y promociones.", "Debe contener @ y dominio. Si no, se marca SIN CORREO REGISTRADO."),
    ("clientes_limpio", "telefono"):          ("Telefono de contacto del cliente.", "9 digitos. Si falta, se rellena con 000000000."),
    ("clientes_limpio", "distrito"):          ("Distrito de residencia declarado.", "Sirve para decidir cobertura de delivery."),
    ("clientes_limpio", "segmento"):          ("Categoria comercial segun frecuencia de compra.", "Nuevo, Recurrente o Frecuente. Define descuentos."),
    ("clientes_limpio", "fecha_registro"):    ("Fecha en que el cliente se afilio a MediSur.", "No puede ser futura."),
    ("clientes_limpio", "condicion_cronica"): ("Enfermedad cronica declarada por el cliente.", "DATO DE SALUD. Se usa solo para alertar interacciones entre medicamentos. Ley 29733."),
    ("clientes_limpio", "correo_fue_corregido"): ("Marca si el ETL de la S10 anulo el correo original por invalido.", "La genera el pipeline, no el usuario. Sirve para auditar la limpieza."),

    ("productos_servicios", "id_producto"):     ("Identificador interno del medicamento o servicio.", "Unico."),
    ("productos_servicios", "nombre_producto"): ("Nombre comercial del medicamento o servicio.", "Es lo que el cliente pide en mostrador."),
    ("productos_servicios", "categoria"):       ("Familia terapeutica o tipo de servicio.", "Agrupa el catalogo y los reportes de venta."),
    ("productos_servicios", "precio_base"):     ("Precio de lista en soles, sin descuentos.", "Debe ser mayor a 0."),
    ("productos_servicios", "activo"):          ("Indica si el producto se sigue vendiendo.", "1 = si, 0 = descontinuado. No se borra: se desactiva."),

    ("ventas_limpia", "id_operacion"):     ("Identificador interno de la venta.", "Unico."),
    ("ventas_limpia", "codigo_operacion"): ("Codigo visible de la venta (O00001).", "Es el que figura en el ticket."),
    ("ventas_limpia", "id_cliente"):       ("Cliente que realizo la compra.", "Debe existir en clientes_limpio (integridad referencial)."),
    ("ventas_limpia", "fecha_operacion"):  ("Fecha en que se realizo la venta.", "No puede ser futura."),
    ("ventas_limpia", "canal"):            ("Medio por el que se hizo la venta.", "Mostrador, App, Web o Telefono."),
    ("ventas_limpia", "monto_total"):      ("Monto cobrado en soles.", "No puede ser negativo."),
    ("ventas_limpia", "monto_fue_ajustado"): ("Marca si el ETL de la S10 corrigio un monto negativo.", "La genera el pipeline. Evidencia de la limpieza."),
    ("ventas_limpia", "cliente_valido"):   ("Marca si la venta apunta a un cliente que existe.", "False = venta huerfana detectada en la S10."),

    ("detalle_operacion", "id_detalle"): ("Identificador interno del ítem o fila en la transacción.", "Único, clave primaria auto-incremental de la tabla."),
    ("detalle_operacion", "id_operacion"): ("Identificador de la venta a la que pertenece el detalle.", "Clave foránea que referencia a ventas_limpia(id_operacion)."),
    ("detalle_operacion", "id_producto"): ("Identificador del producto o servicio vendido en la línea.", "Clave foránea que referencia a productos_servicios(id_producto)."),
    ("detalle_operacion", "cantidad"): ("Número de unidades vendidas del producto en el fila.", "Entero positivo mayor a 0 (cantidad >= 1)."),
    ("detalle_operacion", "precio_unitario"): ("Precio por unidad cobrado al momento de la venta en soles.", "Monto numérico mayor a 0, guardado con 2 decimales."),
    ("detalle_operacion", "subtotal"): ("Importe total correspondiente a la línea o ítem de venta.", "Debe ser igual al producto de cantidad por precio_unitario (subtotal = cantidad * precio_unitario)."),
}


def aplicar_metadatos_negocio(mt):
    """Agrega descripcion y regla de negocio a cada columna del inventario tecnico."""
    mt = mt.copy()
    mt["descripcion_negocio"] = [
        NEGOCIO.get((t, c), ("PENDIENTE DE DOCUMENTAR", ""))[0]
        for t, c in zip(mt["tabla"], mt["columna"])
    ]
    mt["regla_negocio"] = [
        NEGOCIO.get((t, c), ("", "PENDIENTE DE DOCUMENTAR"))[1]
        for t, c in zip(mt["tabla"], mt["columna"])
    ]
    return mt


catalogo = aplicar_metadatos_negocio(metadatos_tecnicos)

documentadas = (catalogo["descripcion_negocio"] != "PENDIENTE DE DOCUMENTAR").sum()
print(f"Columnas con descripcion de negocio: {documentadas} de {len(catalogo)}")
print(f"Cobertura del catalogo: {documentadas/len(catalogo)*100:.1f}%")
print("\nEjemplo - clientes_limpio documentada:")
display(catalogo[catalogo["tabla"] == "clientes_limpio"][
    ["columna", "tipo_dato", "descripcion_negocio", "regla_negocio"]])
display(catalogo[catalogo["tabla"] == "detalle_operacion"][
    ["columna", "tipo_dato", "descripcion_negocio", "regla_negocio"]])
display(catalogo[catalogo["tabla"] == "pagos"][
    ["columna", "tipo_dato", "descripcion_negocio", "regla_negocio"]])

Columnas con descripcion de negocio: 32 de 68
Cobertura del catalogo: 47.1%

Ejemplo - clientes_limpio documentada:


,columna,tipo_dato,descripcion_negocio,regla_negocio
0,id_cliente,INTEGER,Identificador interno del cliente en el sistema.,"Unico, no se reutiliza aunque el cliente se re..."
1,codigo_cliente,TEXT,Codigo comercial visible del cliente (C00001).,Formato C + 5 digitos.
2,tipo_documento,TEXT,Tipo de documento de identidad presentado.,"DNI, CE o Pasaporte."
3,num_documento,TEXT,Numero de documento de identidad.,DNI: 8 digitos. Es el identificador legal del ...
4,nombres,TEXT,Nombres del cliente.,Solo letras. Se guarda con la primera en mayus...
5,apellidos,TEXT,Apellidos del cliente.,Solo letras. Se guarda con la primera en mayus...
6,correo,TEXT,Correo de contacto para boletas y promociones.,"Debe contener @ y dominio. Si no, se marca SIN..."
7,telefono,TEXT,Telefono de contacto del cliente.,"9 digitos. Si falta, se rellena con 000000000."
8,distrito,TEXT,Distrito de residencia declarado.,Sirve para decidir cobertura de delivery.
9,segmento,TEXT,Categoria comercial segun frecuencia de compra.,"Nuevo, Recurrente o Frecuente. Define descuentos."


,columna,tipo_dato,descripcion_negocio,regla_negocio
13,id_detalle,INTEGER,Identificador interno del ítem o fila en la tr...,"Único, clave primaria auto-incremental de la t..."
14,id_operacion,INTEGER,Identificador de la venta a la que pertenece e...,Clave foránea que referencia a ventas_limpia(i...
15,id_producto,INTEGER,Identificador del producto o servicio vendido ...,Clave foránea que referencia a productos_servi...
16,cantidad,INTEGER,Número de unidades vendidas del producto en el...,Entero positivo mayor a 0 (cantidad >= 1).
17,precio_unitario,REAL,Precio por unidad cobrado al momento de la ven...,"Monto numérico mayor a 0, guardado con 2 decim..."
18,subtotal,REAL,Importe total correspondiente a la línea o íte...,Debe ser igual al producto de cantidad por pre...


,columna,tipo_dato,descripcion_negocio,regla_negocio
36,id_pago,INTEGER,PENDIENTE DE DOCUMENTAR,PENDIENTE DE DOCUMENTAR
37,codigo_pago,TEXT,PENDIENTE DE DOCUMENTAR,PENDIENTE DE DOCUMENTAR
38,id_operacion,INTEGER,PENDIENTE DE DOCUMENTAR,PENDIENTE DE DOCUMENTAR
39,fecha_pago,TEXT,PENDIENTE DE DOCUMENTAR,PENDIENTE DE DOCUMENTAR
40,medio_pago,TEXT,PENDIENTE DE DOCUMENTAR,PENDIENTE DE DOCUMENTAR
41,monto_pagado,REAL,PENDIENTE DE DOCUMENTAR,PENDIENTE DE DOCUMENTAR
42,estado_pago,TEXT,PENDIENTE DE DOCUMENTAR,PENDIENTE DE DOCUMENTAR


**Pregunta 2:** El catálogo quedó incompleto a propósito: varias columnas dicen `PENDIENTE DE DOCUMENTAR`. Elige **3 columnas pendientes** de la tabla `pagos` o `detalle_operacion`, y redacta su descripción de negocio y su regla. Escríbelas abajo y agrégalas al diccionario `NEGOCIO`.

Respuesta: Se eligen las columnas id_detalle, id_operacion e id_producto de la tabla detalle_operación.

| Columna | Descripción de negocio | Regla de negocio |
| --- | --- | --- |
| id_detalle | Identificador interno del ítem o fila en la transacción. | Único, clave primaria auto-incremental de la tabla. |
| id_operacion | Identificador de la venta a la que pertenece el detalle. | Clave foránea que referencia a ventas_limpia(id_operacion). |
| id_producto | Identificador del producto o servicio vendido en la fila. | Clave foránea que referencia a productos_servicios(id_producto). |

### Paso 4: clasificación de SENSIBILIDAD

Este es el paso que convierte al catálogo en algo con poder real. Vamos a etiquetar cada columna con su nivel: **Público**, **Interno**, **Confidencial** o **Sensible**.

Hasta aquí el catálogo era documentación. Desde el Paso 8, esta columna va a decidir **qué ve cada rol**. Una columna mal clasificada es una fuga de datos.

### Cada negocio tiene su propio dato delicado

Los 15 casos del curso tienen la misma estructura, **salvo la última columna de `clientes`**: ahí cada empresa guarda lo suyo. MediSur guarda `condicion_cronica`; el banco, `saldo_cuenta`; la universidad, `promedio_notas`.

Y ojo con la ley, porque aquí hay una sorpresa. La **Ley 29733** peruana considera dato sensible no solo la salud, sino también los **ingresos económicos**, el origen racial y étnico, las convicciones políticas o religiosas, la afiliación sindical y los datos biométricos. El **GDPR** europeo *no* incluye los ingresos económicos. Es decir: **en el Perú, el saldo de una cuenta bancaria está tan protegido como un diagnóstico médico.**

Por eso el diccionario `DATO_DELICADO_DEL_CASO` clasifica `saldo_cuenta`, `monto_asegurado` y `linea_credito` como Sensible: no es una opinión, es lo que dice la ley peruana.

Identificar cuál es el dato más delicado de tu negocio no es trabajo del programador: es del **Data Owner**. El programador solo lo obedece.

In [ ]:
# ============================================================
# Paso 4: clasificacion de SENSIBILIDAD de cada columna
# ============================================================
# Sensible     -> categoria especial protegida por ley (salud). Ley 29733 / GDPR.
# Confidencial -> dato personal identificable (PII).
# Interno      -> uso interno de la empresa.
# Publico      -> puede mostrarse a cualquiera.

# ------------------------------------------------------------------
# El dato delicado de CADA negocio.
# Los 15 casos del curso tienen la misma estructura, salvo la ultima columna de
# clientes: ahi cada empresa guarda lo suyo. Identificar cual es el dato mas
# delicado de TU negocio es trabajo del Data Owner, no del programador.
#
# Ojo con la Ley 29733: en el Peru los INGRESOS ECONOMICOS son dato sensible,
# al mismo nivel que la salud. El GDPR europeo no los incluye. Por eso el banco
# y la aseguradora protegen su columna con el mismo rigor que la farmacia.
# ------------------------------------------------------------------
DATO_DELICADO_DEL_CASO = {
    "01_clinica_vitalsalud":   ("diagnostico",             "Sensible"),      # salud
    "02_banco_crediandes":     ("saldo_cuenta",            "Sensible"),      # ingresos economicos
    "03_farmacia_medisur":     ("condicion_cronica",       "Sensible"),      # salud
    "04_vet_cuidapatas":       ("motivo_atencion_mascota", "Interno"),       # dato de la mascota
    "05_hotel_andes_stay":     ("nacionalidad",            "Confidencial"),  # puede revelar origen
    "06_logiexpress_urbano":   ("direccion_entrega",       "Confidencial"),  # ubicacion del cliente
    "07_universidad_aula360":  ("promedio_notas",          "Confidencial"),  # rendimiento academico
    "08_muni_data":            ("tipo_tramite",            "Interno"),       # dato administrativo
    "09_retail_online_market": ("tarjeta_terminacion",     "Confidencial"),  # financiero enmascarado
    "10_agroexport_sol":       ("pais_destino",            "Interno"),       # dato comercial
    "11_energyhome_servicios": ("consumo_kwh",             "Confidencial"),  # revela habitos del hogar
    "12_seguros_protege":      ("monto_asegurado",         "Sensible"),      # ingresos economicos
    "13_fastfood_norte":       ("direccion_delivery",      "Confidencial"),  # ubicacion del cliente
    "14_manufactura_metaltec": ("linea_credito",           "Sensible"),      # ingresos economicos
    "15_telconorte_movil":     ("numero_linea",            "Confidencial"),  # dato de contacto
}

SENSIBILIDAD = {
    "num_documento":     "Confidencial",  # PII: identifica legalmente a la persona
    "correo":            "Confidencial",  # PII: permite contactar al cliente
    "telefono":          "Confidencial",  # PII: permite contactar al cliente

    # Decision deliberada: nombres y apellidos SI son datos personales, pero el
    # vendedor no puede atender a un cliente sin saber a quien atiende. Se
    # clasifican Interno: uso del personal, nunca salen de la empresa.
    # Clasificar de mas es tan danino como clasificar de menos: si el vendedor
    # no puede hacer su trabajo, terminara pidiendo la clave del Data Owner.
    "nombres":           "Interno",
    "apellidos":         "Interno",

    "nombre_producto":   "Publico",
    "categoria":         "Publico",
    "precio_base":       "Publico",
    "activo":            "Publico",
}

# Sumamos al diccionario el dato delicado del caso que estas trabajando.
columna_del_caso, nivel_del_caso = DATO_DELICADO_DEL_CASO[caso]
SENSIBILIDAD[columna_del_caso] = nivel_del_caso
print(f"El dato delicado de tu caso ({caso}):")
print(f"   columna '{columna_del_caso}'  ->  {nivel_del_caso}")


def clasificar_sensibilidad(columna):
    """Devuelve el nivel de sensibilidad de una columna. Por defecto: Interno."""
    return SENSIBILIDAD.get(columna, "Interno")


catalogo["sensibilidad"] = catalogo["columna"].apply(clasificar_sensibilidad)

print("\nColumnas por nivel de sensibilidad:")
print(catalogo["sensibilidad"].value_counts().to_string())

print("\nLas columnas mas protegidas de la base:")
display(catalogo[catalogo["sensibilidad"].isin(["Sensible", "Confidencial"])][
    ["tabla", "columna", "sensibilidad", "descripcion_negocio"]])

El dato delicado de tu caso (03_farmacia_medisur):
   columna 'condicion_cronica'  ->  Sensible

Columnas por nivel de sensibilidad:
sensibilidad
Interno         58
Confidencial     5
Publico          4
Sensible         1

Las columnas mas protegidas de la base:


,tabla,columna,sensibilidad,descripcion_negocio
3,clientes_limpio,num_documento,Confidencial,Numero de documento de identidad.
6,clientes_limpio,correo,Confidencial,Correo de contacto para boletas y promociones.
7,clientes_limpio,telefono,Confidencial,Telefono de contacto del cliente.
11,clientes_limpio,condicion_cronica,Sensible,Enfermedad cronica declarada por el cliente.
23,empleados,num_documento,Confidencial,PENDIENTE DE DOCUMENTAR
26,empleados,correo,Confidencial,PENDIENTE DE DOCUMENTAR


**Pregunta 3:** `condicion_cronica` se clasificó como **Sensible** y `num_documento` como **Confidencial**, aunque ambos son datos personales. ¿Por qué no están en el mismo nivel? Piensa en el daño concreto que sufriría un cliente de MediSur si cada uno se filtrara.

Respuesta: Porque la condición crónica revela el estado de salud y está protegida por la Ley de Protección de Datos Personales. Su filtración generaría un daño grave, como la discriminación laboral, la negativa de pólizas de seguros o el daño psicológico por la exposición pública de la vida privada. El número de documento identifica al cliente para el registro de la venta, pero no lo expone. Su filtración facilita riesgos patrimoniales o suplantación de identidad.

**Pregunta 3b:** Busca tu caso en el diccionario `DATO_DELICADO_DEL_CASO` y mira el de `02_banco_crediandes`: la columna `saldo_cuenta` está clasificada como **Sensible**, el mismo nivel que un diagnóstico médico. ¿Te parece exagerado? Justifica según la Ley 29733 y explica qué pasaría con esa misma columna bajo el GDPR europeo.

Respuesta: No me parece exagerado, ya que los ingresos económicos forman expresamente parte de los datos sensibles protegidos por la ley 29733 para evitar que su uso indebido genere exclusión social o acoso comercial. El saldo bancario se trata bajo las reglas generales de los datos personales comunes (artículo 6 1b del RGPD), requiriendo una base legítima, pero sin la prohibición estricta ni los requisitos reforzados que sí aplican a los datos personales de categoría especial(artículo 9 RGPD).

### Paso 5: metadatos OPERACIONALES y LINAJE

El linaje (o trazabilidad) responde: **¿de dónde salió este dato y qué le hicieron en el camino?**

Es la pregunta que hace todo auditor. Si el gerente ve que un cliente aparece con `SIN CORREO REGISTRADO`, quiere saber si el cliente nunca dio su correo o si un proceso se lo borró. Sin linaje, no hay respuesta.

Tú tienes una ventaja: sabes exactamente qué le hizo tu ETL de la Semana 10 a cada columna. Vamos a dejarlo escrito antes de que se te olvide.

In [ ]:
# ============================================================
# Paso 5: metadatos OPERACIONALES y linaje (origen -> ETL -> destino)
# ============================================================
# Columnas que el pipeline de la S10 modifico o creo.
TRANSFORMADAS_S10 = {
    "correo":               "Los correos sin formato valido se anularon y se marcaron como SIN CORREO REGISTRADO.",
    "telefono":             "Los telefonos vacios se rellenaron con 000000000.",
    "nombres":              "Se quitaron espacios sobrantes y se paso a formato Titulo.",
    "apellidos":            "Se quitaron espacios sobrantes y se paso a formato Titulo.",
    "distrito":             "Se quitaron espacios sobrantes y se paso a formato Titulo.",
    "segmento":             "Se quitaron espacios sobrantes y se paso a formato Titulo.",
    "num_documento":        "Se eliminaron los registros con DNI duplicado (se conservo el primero).",
    "correo_fue_corregido": "COLUMNA NUEVA creada por el ETL de la S10.",
    "monto_ajustado":       "COLUMNA NUEVA creada por el ETL de la S10.",
    "monto_fue_ajustado":   "COLUMNA NUEVA creada por el ETL de la S10.",
    "cliente_valido":       "COLUMNA NUEVA creada por el ETL de la S10.",
    "empleado_valido":      "COLUMNA NUEVA creada por el ETL de la S10.",
}

# Tablas migradas por el ETL vs tablas copiadas tal cual
TABLAS_TRANSFORMADAS = {"clientes_limpio", "ventas_limpia"}


def construir_linaje(fila):
    """Arma la ruta que siguio el dato: origen -> proceso -> destino."""
    if fila["tabla"] in TABLAS_TRANSFORMADAS:
        proceso = "ETL Semana 10 (pipeline de migracion)"
    else:
        proceso = "Copia directa (fase 2 de la migracion, Semana 11)"
    return f"{DB_ORIGEN} -> {proceso} -> {DB_DESTINO}.{fila['tabla']}"


def buscar_transformacion(tabla, columna):
    """Que le hizo el pipeline a esta columna, EN ESTA TABLA.

    Ojo con el detalle: la transformacion depende de la tabla, no solo del nombre
    de la columna. El ETL de la S10 limpio los correos de clientes_limpio, pero
    NUNCA toco la tabla empleados, que tambien tiene una columna 'correo'.
    Si buscaramos solo por nombre de columna, el catalogo afirmaria que los
    correos de empleados fueron limpiados. Seria falso, y un catalogo que miente
    sobre el linaje es peor que no tener catalogo.
    """
    if tabla not in TABLAS_TRANSFORMADAS:
        return "Sin transformacion: se copio tal cual desde el origen."
    return TRANSFORMADAS_S10.get(columna, "Sin transformacion: se migro tal cual.")


catalogo["origen"] = DB_ORIGEN
catalogo["linaje"] = catalogo.apply(construir_linaje, axis=1)
catalogo["transformacion_aplicada"] = [
    buscar_transformacion(t, c) for t, c in zip(catalogo["tabla"], catalogo["columna"])
]
catalogo["fecha_catalogacion"] = "2026-07-15"
catalogo["responsable"] = "Equipo de datos MediSur"

tocadas = catalogo["transformacion_aplicada"].str.startswith("Sin transformacion") == False
print(f"Columnas que el ETL de la S10 toco o creo: {tocadas.sum()}")

print("\nEl linaje de las columnas que tu pipeline modifico:")
display(catalogo[tocadas][["tabla", "columna", "transformacion_aplicada"]])

# Comprobacion: 'correo' existe en dos tablas, pero el ETL solo limpio una.
print("\nLa MISMA columna 'correo' en dos tablas distintas:")
display(catalogo[catalogo["columna"] == "correo"][
    ["tabla", "columna", "transformacion_aplicada"]])

Columnas que el ETL de la S10 toco o creo: 12

El linaje de las columnas que tu pipeline modifico:


,tabla,columna,transformacion_aplicada
3,clientes_limpio,num_documento,Se eliminaron los registros con DNI duplicado ...
4,clientes_limpio,nombres,Se quitaron espacios sobrantes y se paso a for...
5,clientes_limpio,apellidos,Se quitaron espacios sobrantes y se paso a for...
6,clientes_limpio,correo,Los correos sin formato valido se anularon y s...
7,clientes_limpio,telefono,Los telefonos vacios se rellenaron con 000000000.
8,clientes_limpio,distrito,Se quitaron espacios sobrantes y se paso a for...
9,clientes_limpio,segmento,Se quitaron espacios sobrantes y se paso a for...
12,clientes_limpio,correo_fue_corregido,COLUMNA NUEVA creada por el ETL de la S10.
64,ventas_limpia,monto_ajustado,COLUMNA NUEVA creada por el ETL de la S10.
65,ventas_limpia,monto_fue_ajustado,COLUMNA NUEVA creada por el ETL de la S10.



La MISMA columna 'correo' en dos tablas distintas:


,tabla,columna,transformacion_aplicada
6,clientes_limpio,correo,Los correos sin formato valido se anularon y s...
26,empleados,correo,Sin transformacion: se copio tal cual desde el...


**Pregunta 4:** Un auditor revisa la base y encuentra un cliente con el correo `SIN CORREO REGISTRADO`. Usando la columna `transformacion_aplicada` del catálogo, ¿qué le responderías sobre lo que le pasó a ese dato? ¿Y qué habrías respondido si el catálogo no existiera?

Respuesta: De acuerdo a la columna transformación_aplicada, se le diría al auditor que los correos sin formato valido se anularon y por esa razón se marcaron de esa manera. Sin un catálogo, o las direcciones de correo no se hubieran transformado, o tocaría expresar la causa más probable sin una certeza absoluta.

**Pregunta 4b:** Mira la última tabla que imprimió la celda: la columna `correo` existe en `clientes_limpio` **y** en `empleados`, pero el catálogo dice cosas distintas de cada una. ¿Por qué? Lee el comentario de la función `buscar_transformacion()` y explica qué pasaría si el linaje se buscara solo por el nombre de la columna, sin mirar la tabla.

Respuesta: Porque los metadatos técnicos de cada columna son diferentes, además, los correos de la tabla de empleados corresponden a emplados, con lo que los metadatos de negocio también cambian. El ETL solamente se aplicó a la tabla de clientes, por lo que se tiene ahora clientes_limpio. La tabla de empleados necesita pasar por un proceso similar. Si el linaje se buscara sólo por el nombre de la columna, sin mirar la tabla se producirían varias consecuencias, como falsos positivos en el análisis de impacto, o la mezcla del uso comercial del correo del cliente con el uso administrativo del correo del empleado a causa de un linaje ciego.

### Paso 6: guardar el catálogo DENTRO de la base

El catálogo vive en memoria. Si cierras Colab, se pierde.

Un catálogo de datos en un Excel en la laptop de alguien no sirve: se desactualiza, se copia mal y nadie lo encuentra. Lo guardamos como una tabla más, `catalogo_metadatos`, dentro de `farmacia_migrada.db`.

Esto tiene una consecuencia elegante: **la base pasa a describirse a sí misma**. Quien reciba el archivo `.db` recibe los datos y su documentación en el mismo lugar, y puede consultar el catálogo con SQL igual que cualquier otra tabla.

In [ ]:
# ============================================================
# Paso 6: guardar el catalogo unificado como una tabla de la base
# ============================================================
COLUMNAS_CATALOGO = [
    "tabla", "columna", "tipo_dato", "admite_nulos", "es_clave_primaria",
    "descripcion_negocio", "regla_negocio", "sensibilidad",
    "origen", "linaje", "transformacion_aplicada",
    "registros_tabla", "fecha_catalogacion", "responsable",
]

catalogo_final = catalogo[COLUMNAS_CATALOGO]
catalogo_final.to_sql("catalogo_metadatos", conn, if_exists="replace", index=False)
conn.commit()

print("Catalogo guardado dentro de la base como la tabla 'catalogo_metadatos'.")
print("Filas del catalogo:", len(catalogo_final))

# Comprobamos que ahora la base se describe a si misma
tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print("\nTablas de la base del proyecto:")
for t in tablas["name"]:
    marca = "  <-- el catalogo" if t == "catalogo_metadatos" else ""
    print("   -", t, marca)

print("\nAsi se ve el catalogo por dentro:")
display(pd.read_sql_query("SELECT * FROM catalogo_metadatos LIMIT 5", conn))

Catalogo guardado dentro de la base como la tabla 'catalogo_metadatos'.
Filas del catalogo: 68

Tablas de la base del proyecto:
   - catalogo_metadatos   <-- el catalogo
   - clientes_limpio 
   - detalle_operacion 
   - empleados 
   - incidencias 
   - pagos 
   - productos_servicios 
   - sedes 
   - ventas_limpia 

Asi se ve el catalogo por dentro:


,tabla,columna,tipo_dato,admite_nulos,es_clave_primaria,descripcion_negocio,regla_negocio,sensibilidad,origen,linaje,transformacion_aplicada,registros_tabla,fecha_catalogacion,responsable
0,clientes_limpio,id_cliente,INTEGER,Si,No,Identificador interno del cliente en el sistema.,"Unico, no se reutiliza aunque el cliente se re...",Interno,farmacia_legacy.db,farmacia_legacy.db -> ETL Semana 10 (pipeline ...,Sin transformacion: se migro tal cual.,202,2026-07-15,Equipo de datos MediSur
1,clientes_limpio,codigo_cliente,TEXT,Si,No,Codigo comercial visible del cliente (C00001).,Formato C + 5 digitos.,Interno,farmacia_legacy.db,farmacia_legacy.db -> ETL Semana 10 (pipeline ...,Sin transformacion: se migro tal cual.,202,2026-07-15,Equipo de datos MediSur
2,clientes_limpio,tipo_documento,TEXT,Si,No,Tipo de documento de identidad presentado.,"DNI, CE o Pasaporte.",Interno,farmacia_legacy.db,farmacia_legacy.db -> ETL Semana 10 (pipeline ...,Sin transformacion: se migro tal cual.,202,2026-07-15,Equipo de datos MediSur
3,clientes_limpio,num_documento,TEXT,Si,No,Numero de documento de identidad.,DNI: 8 digitos. Es el identificador legal del ...,Confidencial,farmacia_legacy.db,farmacia_legacy.db -> ETL Semana 10 (pipeline ...,Se eliminaron los registros con DNI duplicado ...,202,2026-07-15,Equipo de datos MediSur
4,clientes_limpio,nombres,TEXT,Si,No,Nombres del cliente.,Solo letras. Se guarda con la primera en mayus...,Interno,farmacia_legacy.db,farmacia_legacy.db -> ETL Semana 10 (pipeline ...,Se quitaron espacios sobrantes y se paso a for...,202,2026-07-15,Equipo de datos MediSur


**Pregunta 5:** ¿Qué ventaja concreta tiene guardar el catálogo **dentro de la misma base** en lugar de en un Excel aparte? Menciona al menos dos situaciones reales del equipo de MediSur donde esa decisión marque la diferencia.

Respuesta: Guardar el catálogo dentro de la misma base de datos, como una tabla ejecutable en lugar de un archivo Excel independiente ofrece la ventaja concreta de permitir que la gobernanza de datos y el control de acceso sean dinámicos, automatizados e inseparables de los datos reales. Una situación en la que esto marca la diferencia es el control de acceso en tiempo real según el rol, algo que no se puede hacer en Excel. Otra situación es la continuidad operativa y trazabilidad ETL cuando cambia el personal, que es una ventaja respecto a Excel, en donde la documentación se puede perder o quedar inaccesible a un nuevo desarrollador.

### Paso 7: describir el conjunto con un estándar (Dublin Core)

Hasta ahora describimos **columnas**. Falta describir **el conjunto completo**: quién lo creó, de qué trata, quién es el responsable, qué derechos de uso tiene.

Podríamos inventar nuestros propios campos, pero entonces nadie fuera de MediSur entendería nuestra ficha. Para eso existen los estándares:

- **Dublin Core**: 15 campos para describir cualquier recurso de información. Es simple y universal.
- **ISO 11179**: norma internacional para registrar y nombrar elementos de datos. Es más estricta y se usa en gobiernos y banca.

Usar un estándar es lo que permite que tu catálogo se entienda fuera de tu equipo. Es la diferencia entre documentar y **comunicar**.

In [ ]:
# ============================================================
# Paso 7: ficha Dublin Core del conjunto de datos
# ============================================================
n_tablas = catalogo_final["tabla"].nunique()
n_cols   = len(catalogo_final)
n_sens   = (catalogo_final["sensibilidad"] == "Sensible").sum()

dublin_core = {
    "Title":       "Base de datos operativa de Farmacia MediSur (migrada)",
    "Creator":     "Equipo de datos MediSur - Curso Fundamentos de Gestion de Datos, TECSUP",
    "Subject":     "Farmacia; clientes; ventas; medicamentos; gestion de datos",
    "Description": (f"Base migrada y limpiada del sistema legacy de Farmacia MediSur. "
                    f"Contiene {n_tablas} tablas y {n_cols} columnas documentadas, "
                    f"de las cuales {n_sens} son datos de salud."),
    "Publisher":   "Farmacia MediSur",
    "Contributor": "Pipeline ETL Semana 10; catalogacion Semana 11",
    "Date":        "2026-07-15",
    "Type":        "Dataset",
    "Format":      "SQLite 3 (.db)",
    "Identifier":  DB_DESTINO,
    "Source":      DB_ORIGEN,
    "Language":    "es-PE",
    "Relation":    "Deriva de la base oficial del caso 03_farmacia_medisur",
    "Coverage":    "Peru; operaciones 2025-2026",
    "Rights":      "Uso academico restringido. Contiene datos de salud protegidos por la Ley 29733.",
}

ficha = pd.DataFrame(list(dublin_core.items()), columns=["campo_dublin_core", "valor"])
ficha.to_sql("ficha_dublin_core", conn, if_exists="replace", index=False)
conn.commit()

print("Ficha Dublin Core del conjunto de datos:\n")
for campo, valor in dublin_core.items():
    print(f"  {campo:12s}: {valor}")

Ficha Dublin Core del conjunto de datos:

  Title       : Base de datos operativa de Farmacia MediSur (migrada)
  Creator     : Equipo de datos MediSur - Curso Fundamentos de Gestion de Datos, TECSUP
  Subject     : Farmacia; clientes; ventas; medicamentos; gestion de datos
  Description : Base migrada y limpiada del sistema legacy de Farmacia MediSur. Contiene 8 tablas y 68 columnas documentadas, de las cuales 1 son datos de salud.
  Publisher   : Farmacia MediSur
  Contributor : Pipeline ETL Semana 10; catalogacion Semana 11
  Date        : 2026-07-15
  Type        : Dataset
  Format      : SQLite 3 (.db)
  Identifier  : farmacia_migrada.db
  Source      : farmacia_legacy.db
  Language    : es-PE
  Relation    : Deriva de la base oficial del caso 03_farmacia_medisur
  Coverage    : Peru; operaciones 2025-2026
  Rights      : Uso academico restringido. Contiene datos de salud protegidos por la Ley 29733.


**Pregunta 6:** ¿Para qué sirve usar un estándar como Dublin Core en vez de inventar tus propios campos? Da un ejemplo concreto: imagina que MediSur se fusiona con otra cadena de farmacias y hay que unir los dos catálogos.

Respuesta: Usar un estándar como Dublin Core sirve para garantizar la reutilización y la compatibilidad técnica entre sistemas heterogéneos sin necesidad de rediseñar las estructuras desde cero. Para el ejemplo concreto, si las dos cadenas se unen y no usaran un estándar, los equipos de datos de ambas cadenas tendrían que comparar campo por campo y/o reescribir el código para soportar nuevas equivalencias, algo que no pasaría con el uso de un estándar de catalogación.

### Paso 8: los ROLES — el catálogo empieza a gobernar

Este es el paso más importante del laboratorio.

MediSur tiene cinco tipos de usuario. Cada uno necesita ver cosas distintas:

| Rol | Qué hace en la farmacia | Hasta qué nivel puede ver |
|---|---|---|
| **Vendedor** | Atiende en mostrador | Interno |
| **Cajero** | Cobra las ventas | Interno |
| **Supervisor** | Dirige la sede | Confidencial |
| **Auditor** | Fiscaliza el cumplimiento legal | Sensible |
| **Data Owner** | Es el dueño responsable del dato | Sensible |

Fíjate en lo que **no** vamos a hacer: no vamos a escribir "el vendedor no ve `condicion_cronica`". Eso sería quemar la regla en el código, y el día que aparezca una columna sensible nueva, habría que acordarse de agregarla.

En vez de eso, la función pregunta **al catálogo** qué nivel tiene cada columna, y compara con el nivel del rol. Si mañana alguien cataloga una columna nueva como Sensible, el vendedor deja de verla **automáticamente**, sin tocar una sola línea de código.

Eso es lo que significa que los metadatos gobiernen el sistema.

> **Nota sobre SQLite:** motores como PostgreSQL o SQL Server tienen permisos reales (`GRANT`, `REVOKE`: el lenguaje DCL que viste en la Semana 9). SQLite no los tiene, así que los simulamos en Python. El concepto es el mismo; cambia la herramienta. En la Semana 14 ampliaremos estos roles con permisos de escritura y el marco DAMA.

In [ ]:
# ============================================================
# Paso 8: roles de acceso gobernados por el catalogo
# ============================================================
# Cada rol ve hasta cierto nivel de sensibilidad. En la S11 todos son de SOLO LECTURA.
NIVEL_ACCESO = {
    "Vendedor":   {"Publico", "Interno"},
    "Cajero":     {"Publico", "Interno"},
    "Supervisor": {"Publico", "Interno", "Confidencial"},
    "Auditor":    {"Publico", "Interno", "Confidencial", "Sensible"},
    "DataOwner":  {"Publico", "Interno", "Confidencial", "Sensible"},
    "PracticanteMarketing": {"Publico", "Interno"}, #Nuevo rol
}


def columnas_visibles(tabla, rol):
    """Le pregunta AL CATALOGO que columnas de esta tabla puede ver este rol.

    Aqui no hay ninguna columna escrita a mano: la decision sale de los metadatos.
    """
    permitidos = NIVEL_ACCESO.get(rol, set())
    if not permitidos:
        return []
    cat = pd.read_sql_query(
        "SELECT columna, sensibilidad FROM catalogo_metadatos WHERE tabla = ?",
        conn, params=(tabla,))
    return [c for c, s in zip(cat["columna"], cat["sensibilidad"]) if s in permitidos]


def consultar(tabla, rol, limite=10):
    """Devuelve la tabla con SOLO las columnas que el rol tiene permitido ver."""
    cols = columnas_visibles(tabla, rol)
    if not cols:
        return f"ACCESO DENEGADO: el rol {rol} no puede consultar {tabla}."
    lista = ", ".join(cols)
    return pd.read_sql_query(f"SELECT {lista} FROM {tabla} LIMIT {limite}", conn)


# --- La demostracion: la MISMA consulta, dos roles, dos resultados
print("=" * 70)
print("El VENDEDOR consulta los clientes:")
print("=" * 70)
display(consultar("clientes_limpio", "Vendedor", 5))

print("=" * 70)
print("El AUDITOR consulta los MISMOS clientes:")
print("=" * 70)
display(consultar("clientes_limpio", "Auditor", 5))

print("\nColumnas que ve el Vendedor:", len(columnas_visibles("clientes_limpio", "Vendedor")))
print("Columnas que ve el Auditor :", len(columnas_visibles("clientes_limpio", "Auditor")))
print("\nEl vendedor NO ve:", sorted(set(columnas_visibles("clientes_limpio", "Auditor")) -
                                     set(columnas_visibles("clientes_limpio", "Vendedor"))))

El VENDEDOR consulta los clientes:


,id_cliente,codigo_cliente,tipo_documento,nombres,apellidos,distrito,segmento,fecha_registro,correo_fue_corregido
0,1,C0001,Ruc,Iván,Aguirre Bautista,Rímac,Cliente Frecuente,2026-01-24,0
1,2,C0002,Dni,Melissa,Torres Lévano,Villa El Salvador,Paciente Crónico,2026-05-18,0
2,3,C0003,Dni,Milagros,Villanueva Zevallos,Surco,Compra Ocasional,2026-03-08,0
3,4,C0004,Dni,Karla,Ríos Ninaquispe,Villa El Salvador,Convenio Corporativo,2026-05-10,0
4,5,C0005,Dni,Hugo,Romero Castillo,Surco,Cliente Frecuente,2026-04-20,0


El AUDITOR consulta los MISMOS clientes:


,id_cliente,codigo_cliente,tipo_documento,num_documento,nombres,apellidos,correo,telefono,distrito,segmento,fecha_registro,condicion_cronica,correo_fue_corregido
0,1,C0001,Ruc,79941052,Iván,Aguirre Bautista,iván1@correo.com,923380330,Rímac,Cliente Frecuente,2026-01-24,Hipertensión,0
1,2,C0002,Dni,377593,Melissa,Torres Lévano,melissa2@correo.com,966971049,Villa El Salvador,Paciente Crónico,2026-05-18,Asma,0
2,3,C0003,Dni,38119160,Milagros,Villanueva Zevallos,milagros3@correo.com,911285064,Surco,Compra Ocasional,2026-03-08,Ninguna,0
3,4,C0004,Dni,14282620,Karla,Ríos Ninaquispe,karla4@correo.com,999382396,Villa El Salvador,Convenio Corporativo,2026-05-10,Ninguna,0
4,5,C0005,Dni,12191834,Hugo,Romero Castillo,hugo5@correo.com,982000067,Surco,Cliente Frecuente,2026-04-20,Asma,0



Columnas que ve el Vendedor: 9
Columnas que ve el Auditor : 13

El vendedor NO ve: ['condicion_cronica', 'correo', 'num_documento', 'telefono']


**Pregunta 7:** La función `columnas_visibles()` no menciona ni una sola vez la columna `condicion_cronica`, y sin embargo el vendedor no la ve. Explica con tus palabras cómo lo logra. Luego responde: si mañana MediSur agrega una columna `alergias` y la catalogas como Sensible, ¿habría que modificar el código para que el vendedor no la vea? ¿Por qué?

Respuesta: La función lo logra filtrando las columnas visibles de una tabla según los permisos del rol del usuario al hacer una consulta SQL al catálogo de metadatos(catalogo_final) y compara la sensibilidad de cada columna con los niveles permitidos definidos en el diccionario NIVEL_ACCESO.
Si se agrega una nueva columna con datos sensibles no habría que modificar el código, porque el menú ejecuta una consulta SQL dinámica sobre la tabla del Catálogo de Metadatos cada vez que un usuario ingresa al sistema. Al añadir la columna alergias en la tabla física y registrarla en la tabla de catálogo con la etiqueta Sensible, la consulta del menú la excluirá en tiempo de ejecución.

### Paso 9: las funciones del sistema

Ya tenemos el motor. Ahora escribimos las funciones que el usuario va a usar desde el menú, agrupadas en las tres áreas del sistema:

- **Catálogo de productos**: qué medicamentos hay, a qué precio, en qué categoría.
- **Clientes**: ver y buscar, siempre respetando el rol.
- **Catálogo de datos**: el diccionario, las columnas sensibles, el linaje.

Observa que el catálogo de productos **no filtra por rol**: el precio de un medicamento es información pública, cualquiera puede verla. Los clientes sí filtran. No todo dato necesita protección; protegerlo todo por igual es tan malo como no proteger nada.

In [ ]:
# ============================================================
# Paso 9a: CATALOGO DE PRODUCTOS (el catalogo de medicamentos)
# ============================================================
def ver_catalogo_productos():
    """Todo el catalogo de medicamentos y servicios. Es informacion PUBLICA."""
    return pd.read_sql_query("""
        SELECT id_producto, nombre_producto, categoria, precio_base,
               CASE WHEN activo = 1 THEN 'Si' ELSE 'No' END AS a_la_venta
        FROM productos_servicios
        ORDER BY categoria, nombre_producto""", conn)


def buscar_medicamento(texto):
    """Busca un medicamento por nombre. Como cuando el cliente pregunta en mostrador."""
    if not texto or not texto.strip():
        return "Escribe algo para buscar."
    d = pd.read_sql_query("""
        SELECT id_producto, nombre_producto, categoria, precio_base
        FROM productos_servicios
        WHERE nombre_producto LIKE ?
        ORDER BY nombre_producto""", conn, params=(f"%{texto.strip()}%",))
    if len(d) == 0:
        return f"No se encontro ningun medicamento con '{texto}'."
    return d


def ver_categorias():
    """Resumen del catalogo por familia terapeutica."""
    return pd.read_sql_query("""
        SELECT categoria,
               COUNT(*)              AS productos,
               ROUND(MIN(precio_base), 2) AS precio_min,
               ROUND(MAX(precio_base), 2) AS precio_max
        FROM productos_servicios
        GROUP BY categoria
        ORDER BY productos DESC""", conn)


print("--- CATALOGO DE PRODUCTOS (primeros 8) ---")
display(ver_catalogo_productos().head(8))

print("--- Un cliente pregunta en mostrador por paracetamol ---")
display(buscar_medicamento("paracetamol"))

print("--- CATEGORIAS ---")
display(ver_categorias())

--- CATALOGO DE PRODUCTOS (primeros 8) ---


,id_producto,nombre_producto,categoria,precio_base,a_la_venta
0,14,Alprazolam 0.5 mg x 30 tabletas,Controlado,25.11,No
1,13,Clonazepam 0.5 mg x 30 tabletas,Controlado,18.00,Si
2,18,Codeína jarabe 120 ml,Controlado,95.00,No
3,16,Pregabalina 75 mg x 30 cápsulas,Controlado,48.84,Si
4,15,Tramadol 50 mg x 20 cápsulas,Controlado,35.01,Si
5,17,Zolpidem 10 mg x 30 tabletas,Controlado,68.11,Si
6,25,Alcohol medicinal 70% 1 L,Cuidado personal,5.00,No
7,27,Jabón dermatológico 90 g,Cuidado personal,14.20,Si


--- Un cliente pregunta en mostrador por paracetamol ---


,id_producto,nombre_producto,categoria,precio_base
0,2,Paracetamol 500 mg x 100 tabletas,Genérico,10.75


--- CATEGORIAS ---


,categoria,productos,precio_min,precio_max
0,Suplemento,6,16.0,110.0
1,Marca,6,12.0,78.0
2,Genérico,6,8.0,35.0
3,Cuidado personal,6,5.0,68.0
4,Controlado,6,18.0,95.0


In [ ]:
# ============================================================
# Paso 9b: CLIENTES (filtrados por rol, gracias al catalogo)
# ============================================================
def ver_clientes(rol, limite=10):
    """Lista de clientes con las columnas que el rol tiene permitido ver."""
    return consultar("clientes_limpio", rol, limite)


def buscar_cliente(rol, texto):
    """Busca un cliente por nombre o apellido, respetando el rol."""
    cols = columnas_visibles("clientes_limpio", rol)
    if not cols:
        return f"ACCESO DENEGADO: el rol {rol} no puede consultar clientes."
    if not texto or not texto.strip():
        return "Escribe algo para buscar."
    lista = ", ".join(cols)
    d = pd.read_sql_query(
        f"""SELECT {lista} FROM clientes_limpio
            WHERE nombres LIKE ? OR apellidos LIKE ?
            LIMIT 20""",
        conn, params=(f"%{texto.strip()}%", f"%{texto.strip()}%"))
    if len(d) == 0:
        return f"No se encontro ningun cliente con '{texto}'."
    return d


print("--- El SUPERVISOR busca a un cliente ---")
display(buscar_cliente("Supervisor", "a"))

--- El SUPERVISOR busca a un cliente ---


,id_cliente,codigo_cliente,tipo_documento,num_documento,nombres,apellidos,correo,telefono,distrito,segmento,fecha_registro,correo_fue_corregido
0,1,C0001,Ruc,79941052,Iván,Aguirre Bautista,iván1@correo.com,923380330,Rímac,Cliente Frecuente,2026-01-24,0
1,2,C0002,Dni,377593,Melissa,Torres Lévano,melissa2@correo.com,966971049,Villa El Salvador,Paciente Crónico,2026-05-18,0
2,3,C0003,Dni,38119160,Milagros,Villanueva Zevallos,milagros3@correo.com,911285064,Surco,Compra Ocasional,2026-03-08,0
3,4,C0004,Dni,14282620,Karla,Ríos Ninaquispe,karla4@correo.com,999382396,Villa El Salvador,Convenio Corporativo,2026-05-10,0
4,5,C0005,Dni,12191834,Hugo,Romero Castillo,hugo5@correo.com,982000067,Surco,Cliente Frecuente,2026-04-20,0
5,6,C0006,Ruc,16485924,Natalia,Ríos Sánchez,natalia6@correo.com,957360193,None,Paciente Crónico,2026-05-06,0
6,7,C0007,Dni,68567899,Gabriela,Aguirre Chávez,gabriela7@correo.com,995318955,Ate,Compra Ocasional,2026-04-11,0
7,9,C0009,Dni,12055906,Lucía,Romero Rojas,lucía9@correo.com,921860547,San Borja,Cliente Frecuente,2026-03-17,0
8,10,C0010,Dni,55658713,Manuel,Quispe Cruz,manuel10@correo.com,933813783,La Molina,Paciente Crónico,2026-01-17,0
9,11,C0011,Ce,87518432,Luis,Paredes Mendoza,luis11@correo.com,912160684,None,Compra Ocasional,2026-01-01,0


In [ ]:
# ============================================================
# Paso 9c: CATALOGO DE DATOS (el diccionario de la base)
# ============================================================
def ver_diccionario(tabla):
    """Muestra que significa cada columna de una tabla."""
    d = pd.read_sql_query("""
        SELECT columna, tipo_dato, sensibilidad, descripcion_negocio, regla_negocio
        FROM catalogo_metadatos WHERE tabla = ?""", conn, params=(tabla,))
    if len(d) == 0:
        return f"La tabla '{tabla}' no esta en el catalogo."
    return d


def ver_columnas_sensibles(rol):
    """Que columnas guardan datos protegidos. Solo Auditor y Data Owner."""
    if "Sensible" not in NIVEL_ACCESO.get(rol, set()):
        return (f"ACCESO DENEGADO: el rol {rol} no puede consultar el inventario "
                f"de datos sensibles. Solo Auditor y Data Owner.")
    return pd.read_sql_query("""
        SELECT tabla, columna, sensibilidad, descripcion_negocio
        FROM catalogo_metadatos
        WHERE sensibilidad IN ('Sensible', 'Confidencial')
        ORDER BY CASE sensibilidad WHEN 'Sensible' THEN 1 ELSE 2 END, tabla""", conn)


def ver_linaje(columna):
    """De donde vino esta columna y que le hizo el pipeline."""
    d = pd.read_sql_query("""
        SELECT tabla, columna, origen, linaje, transformacion_aplicada
        FROM catalogo_metadatos WHERE columna = ?""", conn, params=(columna,))
    if len(d) == 0:
        return f"La columna '{columna}' no esta en el catalogo."
    return d


def ver_ficha_dublin_core():
    """La ficha estandar del conjunto de datos."""
    return pd.read_sql_query("SELECT * FROM ficha_dublin_core", conn)


def exportar_catalogo():
    """Descarga el catalogo en CSV, para compartirlo fuera del sistema."""
    d = pd.read_sql_query("SELECT * FROM catalogo_metadatos", conn)
    ruta = "catalogo_metadatos_medisur.csv"
    d.to_csv(ruta, index=False, encoding="utf-8-sig")
    return ruta


def tablas_del_catalogo():
    """Lista las tablas documentadas, para el submenu."""
    return pd.read_sql_query(
        "SELECT DISTINCT tabla FROM catalogo_metadatos ORDER BY tabla", conn)["tabla"].tolist()


print("--- Diccionario de productos_servicios ---")
display(ver_diccionario("productos_servicios"))
print("--- El VENDEDOR intenta ver las columnas sensibles ---")
print(ver_columnas_sensibles("Vendedor"))
print("\n--- El AUDITOR ve las columnas sensibles ---")
display(ver_columnas_sensibles("Auditor"))

--- Diccionario de productos_servicios ---


,columna,tipo_dato,sensibilidad,descripcion_negocio,regla_negocio
0,id_producto,INTEGER,Interno,Identificador interno del medicamento o servicio.,Unico.
1,codigo_producto,TEXT,Interno,PENDIENTE DE DOCUMENTAR,PENDIENTE DE DOCUMENTAR
2,nombre_producto,TEXT,Publico,Nombre comercial del medicamento o servicio.,Es lo que el cliente pide en mostrador.
3,categoria,TEXT,Publico,Familia terapeutica o tipo de servicio.,Agrupa el catalogo y los reportes de venta.
4,precio_base,REAL,Publico,"Precio de lista en soles, sin descuentos.",Debe ser mayor a 0.
5,activo,INTEGER,Publico,Indica si el producto se sigue vendiendo.,"1 = si, 0 = descontinuado. No se borra: se des..."


--- El VENDEDOR intenta ver las columnas sensibles ---
ACCESO DENEGADO: el rol Vendedor no puede consultar el inventario de datos sensibles. Solo Auditor y Data Owner.

--- El AUDITOR ve las columnas sensibles ---


,tabla,columna,sensibilidad,descripcion_negocio
0,clientes_limpio,condicion_cronica,Sensible,Enfermedad cronica declarada por el cliente.
1,clientes_limpio,num_documento,Confidencial,Numero de documento de identidad.
2,clientes_limpio,correo,Confidencial,Correo de contacto para boletas y promociones.
3,clientes_limpio,telefono,Confidencial,Telefono de contacto del cliente.
4,empleados,num_documento,Confidencial,PENDIENTE DE DOCUMENTAR
5,empleados,correo,Confidencial,PENDIENTE DE DOCUMENTAR


### Paso 10: EL MENÚ — el sistema toma forma

Tienes las funciones, pero para usarlas hay que saber Python. El gerente de MediSur no sabe Python. Necesita un **menú**: elegir con números.

Un menú son tres piezas, y nada más:

| Pieza | Para qué sirve |
|---|---|
| `while True:` | Repite el menú para siempre, hasta que el usuario decida salir |
| `input()` | Pregunta al usuario qué quiere hacer y espera su respuesta |
| `if / elif / else` | Según lo que eligió, llama a la función correspondiente |
| `break` | Rompe el `while` y sale del menú |

Y como el sistema tiene tres áreas grandes, usamos **submenús**: el menú principal te lleva a un área, y cada área tiene su propio menú con la opción `0) Volver`. Cada submenú es la misma estructura `while True` anidada dentro de la anterior.

Lo importante: **el rol se elige una sola vez, al entrar**, y viaja a todas las funciones. Por eso el vendedor y el auditor recorren exactamente el mismo menú y ven cosas distintas.

Este menú va a crecer todas las semanas: en la 12 le agregarás los gráficos, en la 13 el reporte de calidad, en la 14 los permisos de escritura y la auditoría, y en la 15 quedará completo para el PMD2.

In [ ]:
# ============================================================
# Paso 10: EL MENU con submenus
# ============================================================
ROLES_MENU = {"1": "Vendedor", "2": "Cajero", "3": "Supervisor",
              "4": "Auditor",  "5": "DataOwner", "6": "PracticanteMarketing"} #Nueva opción(6)


def _mostrar(resultado):
    """Muestra bonito, sea un DataFrame o un mensaje de texto."""
    if isinstance(resultado, pd.DataFrame):
        display(resultado)
    else:
        print(resultado)


# ------------------------------------------------- submenu 1: productos
def submenu_productos(rol):
    while True:
        print("\n--- CATALOGO DE PRODUCTOS ---")
        print("  1) Ver todo el catalogo")
        print("  2) Buscar un medicamento")
        print("  3) Ver por categoria")
        print("  0) Volver")
        op = input("  Opcion: ").strip()

        if op == "0":
            break
        elif op == "1":
            _mostrar(ver_catalogo_productos())
        elif op == "2":
            _mostrar(buscar_medicamento(input("  Nombre del medicamento: ")))
        elif op == "3":
            _mostrar(ver_categorias())
        else:
            print("  Opcion invalida.")


# ------------------------------------------------- submenu 2: clientes
def submenu_clientes(rol):
    while True:
        print("\n--- CLIENTES ---")
        print("  1) Ver lista de clientes")
        print("  2) Buscar un cliente")
        print("  0) Volver")
        op = input("  Opcion: ").strip()

        if op == "0":
            break
        elif op == "1":
            _mostrar(ver_clientes(rol))
            print(f"  (Como {rol} ves {len(columnas_visibles('clientes_limpio', rol))} columnas.)")
        elif op == "2":
            _mostrar(buscar_cliente(rol, input("  Nombre o apellido: ")))
        else:
            print("  Opcion invalida.")


# ------------------------------------------------- submenu 3: catalogo de datos
def submenu_catalogo_datos(rol):
    while True:
        print("\n--- CATALOGO DE DATOS (metadatos) ---")
        print("  1) Ver el diccionario de una tabla")
        print("  2) Ver las columnas sensibles      (solo Auditor / Data Owner)")
        print("  3) Ver el linaje de una columna")
        print("  4) Ver la ficha Dublin Core")
        print("  5) Descargar el catalogo en CSV")
        print("  0) Volver")
        op = input("  Opcion: ").strip()

        if op == "0":
            break
        elif op == "1":
            tablas = tablas_del_catalogo()
            print("  Tablas documentadas:")
            for i, t in enumerate(tablas, 1):
                print(f"    {i}) {t}")
            elegida = input("  Numero de la tabla: ").strip()
            if elegida.isdigit() and 1 <= int(elegida) <= len(tablas):
                _mostrar(ver_diccionario(tablas[int(elegida) - 1]))
            else:
                print("  Numero invalido.")
        elif op == "2":
            _mostrar(ver_columnas_sensibles(rol))
        elif op == "3":
            _mostrar(ver_linaje(input("  Nombre de la columna (ej: correo): ").strip()))
        elif op == "4":
            _mostrar(ver_ficha_dublin_core())
        elif op == "5":
            ruta = exportar_catalogo()
            print("  Catalogo exportado:", ruta)
            try:
                from google.colab import files
                files.download(ruta)
            except Exception:
                print("  (Descarga disponible solo en Colab.)")
        else:
            print("  Opcion invalida.")


# ------------------------------------------------- menu principal
def menu():
    print("=" * 52)
    print("     SISTEMA DE DATOS - FARMACIA MEDISUR")
    print("             Semana 11: el catalogo")
    print("=" * 52)
    print("Elige tu rol:")
    print("  1) Vendedor     2) Cajero      3) Supervisor")
    print("  4) Auditor      5) Data Owner  6) Practicante de marketing")

    rol = ROLES_MENU.get(input("Numero de tu rol: ").strip())
    if rol is None:
        print("Rol no valido. Vuelve a ejecutar menu().")
        return

    nivel = ", ".join(sorted(NIVEL_ACCESO[rol]))
    print(f"\nBienvenido, {rol}.")
    print(f"Tu nivel de acceso te permite ver datos: {nivel}")

    while True:
        print("\n" + "=" * 52)
        print(f"MENU PRINCIPAL     (rol: {rol})")
        print("=" * 52)
        print("  1) Catalogo de productos")
        print("  2) Clientes")
        print("  3) Catalogo de datos (metadatos)")
        print("  0) Salir")
        op = input("Opcion: ").strip()

        if op == "0":
            print("Sesion cerrada. Hasta luego,", rol)
            break
        elif op == "1":
            submenu_productos(rol)
        elif op == "2":
            submenu_clientes(rol)
        elif op == "3":
            submenu_catalogo_datos(rol)
        else:
            print("Opcion invalida.")


print("Menu listo. Ejecuta  menu()  en la siguiente celda.")

Menu listo. Ejecuta  menu()  en la siguiente celda.


In [ ]:
# Ejecuta el sistema. Pruebalo dos veces: una como Vendedor (1) y otra como Auditor (4).
menu()

     SISTEMA DE DATOS - FARMACIA MEDISUR
             Semana 11: el catalogo
Elige tu rol:
  1) Vendedor     2) Cajero      3) Supervisor
  4) Auditor      5) Data Owner  6) Practicante de marketing
Numero de tu rol: 6

Bienvenido, PracticanteMarketing.
Tu nivel de acceso te permite ver datos: Interno, Publico

MENU PRINCIPAL     (rol: PracticanteMarketing)
  1) Catalogo de productos
  2) Clientes
  3) Catalogo de datos (metadatos)
  0) Salir
Opcion: 3

--- CATALOGO DE DATOS (metadatos) ---
  1) Ver el diccionario de una tabla
  2) Ver las columnas sensibles      (solo Auditor / Data Owner)
  3) Ver el linaje de una columna
  4) Ver la ficha Dublin Core
  5) Descargar el catalogo en CSV
  0) Volver
  Opcion: 4


,campo_dublin_core,valor
0,Title,Base de datos operativa de Farmacia MediSur (m...
1,Creator,Equipo de datos MediSur - Curso Fundamentos de...
2,Subject,Farmacia; clientes; ventas; medicamentos; gest...
3,Description,Base migrada y limpiada del sistema legacy de ...
4,Publisher,Farmacia MediSur
5,Contributor,Pipeline ETL Semana 10; catalogacion Semana 11
6,Date,2026-07-15
7,Type,Dataset
8,Format,SQLite 3 (.db)
9,Identifier,farmacia_migrada.db



--- CATALOGO DE DATOS (metadatos) ---
  1) Ver el diccionario de una tabla
  2) Ver las columnas sensibles      (solo Auditor / Data Owner)
  3) Ver el linaje de una columna
  4) Ver la ficha Dublin Core
  5) Descargar el catalogo en CSV
  0) Volver
  Opcion: 0

MENU PRINCIPAL     (rol: PracticanteMarketing)
  1) Catalogo de productos
  2) Clientes
  3) Catalogo de datos (metadatos)
  0) Salir
Opcion: 2

--- CLIENTES ---
  1) Ver lista de clientes
  2) Buscar un cliente
  0) Volver
  Opcion: 1


,id_cliente,codigo_cliente,tipo_documento,nombres,apellidos,distrito,segmento,fecha_registro,correo_fue_corregido
0,1,C0001,Ruc,Iván,Aguirre Bautista,Rímac,Cliente Frecuente,2026-01-24,0
1,2,C0002,Dni,Melissa,Torres Lévano,Villa El Salvador,Paciente Crónico,2026-05-18,0
2,3,C0003,Dni,Milagros,Villanueva Zevallos,Surco,Compra Ocasional,2026-03-08,0
3,4,C0004,Dni,Karla,Ríos Ninaquispe,Villa El Salvador,Convenio Corporativo,2026-05-10,0
4,5,C0005,Dni,Hugo,Romero Castillo,Surco,Cliente Frecuente,2026-04-20,0
5,6,C0006,Ruc,Natalia,Ríos Sánchez,None,Paciente Crónico,2026-05-06,0
6,7,C0007,Dni,Gabriela,Aguirre Chávez,Ate,Compra Ocasional,2026-04-11,0
7,8,C0008,Dni,Víctor,Torres Romero,Comas,Convenio Corporativo,2026-05-20,0
8,9,C0009,Dni,Lucía,Romero Rojas,San Borja,Cliente Frecuente,2026-03-17,0
9,10,C0010,Dni,Manuel,Quispe Cruz,La Molina,Paciente Crónico,2026-01-17,0


  (Como PracticanteMarketing ves 9 columnas.)

--- CLIENTES ---
  1) Ver lista de clientes
  2) Buscar un cliente
  0) Volver
  Opcion: 0

MENU PRINCIPAL     (rol: PracticanteMarketing)
  1) Catalogo de productos
  2) Clientes
  3) Catalogo de datos (metadatos)
  0) Salir
Opcion: 0
Sesion cerrada. Hasta luego, PracticanteMarketing


**Pregunta 8:** Ejecuta `menu()` **dos veces**: primero entra como **Vendedor** y luego como **Auditor**. En ambos casos ve a `2) Clientes` → `1) Ver lista`. ¿Cuántas columnas ves con cada rol y cuáles desaparecen? Explica por qué el menú es idéntico pero el resultado no.

Respuesta: Como vendedor se ven 9 columnas y como auditor se ven 13 columnas. Las columnas no visibles para el vendedor son num_documento, correo, telefono y condicion_cronica. La explicación es que el código logra emular satisfactoriamente los permisos GRANT del DCL. Para este ejemplo se llama a la función ver_clientes, que lleva por parámetros el rol y un límite, y retorna el resultado de la función consultar, que a su vez, lleva por parámetros la tabla clientes_limpio, el rol y el límite.

### Paso 11: guardar el avance del proyecto

Cierra la conexión y **descarga `farmacia_migrada.db`**. Esa base ya no es solo datos: lleva dentro su propio catálogo. Es el entregable acumulativo de la Semana 11 y el punto de partida de la Semana 12.

In [ ]:
# ============================================================
# Paso 11: cierre y respaldo del avance
# ============================================================
resumen = pd.read_sql_query("""
    SELECT sensibilidad, COUNT(*) AS columnas
    FROM catalogo_metadatos
    GROUP BY sensibilidad
    ORDER BY columnas DESC""", conn)

print("RESUMEN DEL CATALOGO - SEMANA 11")
print("=" * 40)
print("Tablas documentadas :", catalogo_final["tabla"].nunique())
print("Columnas catalogadas:", len(catalogo_final))
print()
print(resumen.to_string(index=False))

pendientes = pd.read_sql_query("""
    SELECT COUNT(*) AS n FROM catalogo_metadatos
    WHERE descripcion_negocio = 'PENDIENTE DE DOCUMENTAR'""", conn)["n"][0]
print("\nColumnas aun sin documentar:", pendientes, " <-- tu tarea de la Pregunta 2")

conn.close()
print("\nConexion cerrada.")

# Descarga la base del proyecto: la necesitas en la Semana 12.
try:
    from google.colab import files
    files.download(DB_DESTINO)
    print("Descargando", DB_DESTINO)
except Exception:
    print("Guarda manualmente", DB_DESTINO)

RESUMEN DEL CATALOGO - SEMANA 11
Tablas documentadas : 8
Columnas catalogadas: 68

sensibilidad  columnas
     Interno        58
Confidencial         5
     Publico         4
    Sensible         1

Columnas aun sin documentar: 36  <-- tu tarea de la Pregunta 2

Conexion cerrada.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando farmacia_migrada.db


---

## Actividad 3: caso de estudio — el catálogo de MediSur

Responde con lo que construiste hoy. Justifica con datos de tu notebook, no con opiniones.

**Pregunta A:** El gerente pregunta: *"¿Quién puede ver las condiciones crónicas de mis clientes?"*. Responde con tu sistema en la mano: qué roles la ven, cuáles no, y **qué mecanismo** lo garantiza.

Respuesta: Únicamente un auditor y un data owner pueden ver las condiciones crónicas. El mecanismo que lo garantiza empieza por la clasificación de sensibilidad en la tabla de metadatos, sigue en la matriz(diccionario)de permisos por rol NIVEL_ACCESO y termina con el filtro dinámico en el catálogo de metadatos por el menú del sistema que extrae las columnas permitidas y solamente en ellas hace una consulta SQL.

**Pregunta B:** MediSur va a contratar un **practicante de marketing** que necesita ver los distritos y segmentos de los clientes para diseñar promociones, pero no debe ver ningún dato de contacto. Define su nivel de acceso, agrégalo a `NIVEL_ACCESO` y a `ROLES_MENU`, y demuestra que funciona.

Respuesta: Un practicante de marketing debería tener acceso únicamente a datos públicos e internos de las tablas sedes y clientes_limpio. Como su trabajo no encaja en los roles preestablecidos, se debe crear un nuevo rol con los respectivos permisos, similares a los de un vendedor o cajero. Además, hay que agregar el nuevo rol al menú ejecutable. Esos son todos los cambios necesarios.

**Pregunta C:** Un inspector de la autoridad de protección de datos pide el inventario de datos personales y sensibles de MediSur, con su justificación de uso. Genera ese inventario consultando tu catálogo con SQL, y pega la consulta y el resultado.

Respuesta: Se ejecuta una consulta SQL sobre la tabla catalogo_metadatos. La consulta filtra las columnas cuya sensibilidad corresponde a datos protegidos (Sensible y Confidencial), extrayendo su descripción de negocio (justificación de uso) y su regla de negocio asociada (finalidad operativa y marco legal).

**Pregunta D (reto):** Tu catálogo se construyó una sola vez. Si mañana alguien agrega la columna `alergias` a `clientes_limpio` y **no** la cataloga, ¿qué nivel de sensibilidad tendría según `clasificar_sensibilidad()`? ¿Es esa una decisión segura o peligrosa? Propón cómo detectar columnas sin catalogar.

Respuesta: Según clasificar_sensibilidad, si la columna alergias no se registra explícitamente en el diccionario SENSIBILIDAD, el sistema le asignará el nivel Interno por defecto. Es una decisión peligrosa, porque pone los datos al alcance de personas con roles operativos y se incumple la norma de la ley 29733. La forma más sencilla y rápida para tratar columnas desconocidas sería cambiar el parámetro de retorno de SENSIBILIDAD.get(), en la función clasificar_sensibilidad(), de "Interno" a "Sensible", para modificar la condición por defecto que obtiene un dato hasta que el Data Owner lo revise. Otra forma, más compleja, podría ser definir una función que busque periódicamente columnas sin catalogar.

In [ ]:
# Bloque de código adicional para la pregunta 3, actividad 3
query = """
SELECT
    tabla,
    columna,
    sensibilidad,
    descripcion_negocio AS justificacion_de_uso,
    regla_negocio AS marco_o_regla_asociada
FROM catalogo_metadatos
WHERE sensibilidad IN ('Sensible', 'Confidencial')
ORDER BY
    CASE sensibilidad
        WHEN 'Sensible' THEN 1
        WHEN 'Confidencial' THEN 2
    END,
    tabla,
    columna;
"""

df_res = pd.read_sql_query(query, conn)
print(df_res.to_markdown(index=False))

| tabla           | columna           | sensibilidad   | justificacion_de_uso                           | marco_o_regla_asociada                                                               |
|:----------------|:------------------|:---------------|:-----------------------------------------------|:-------------------------------------------------------------------------------------|
| clientes_limpio | condicion_cronica | Sensible       | Enfermedad cronica declarada por el cliente.   | DATO DE SALUD. Se usa solo para alertar interacciones entre medicamentos. Ley 29733. |
| clientes_limpio | correo            | Confidencial   | Correo de contacto para boletas y promociones. | Debe contener @ y dominio. Si no, se marca SIN CORREO REGISTRADO.                    |
| clientes_limpio | num_documento     | Confidencial   | Numero de documento de identidad.              | DNI: 8 digitos. Es el identificador legal del cliente.                               |
| clientes_limpio | telefono       

---

## Actividad final

Redacta tres conclusiones breves:

**1.** ¿Por qué un catálogo de metadatos vale más dentro de la base que en un documento aparte?
 Un catálogo de metadatos dentro de la misma base de datos, vale más que un documento aparte porque el primero permite que la gobernanza de datos y el control de acceso sean dinámicos, automatizados e inseparables de los datos reales.

**2.** ¿Qué significa que "los metadatos gobiernan el sistema"? Explícalo con lo que viste al comparar el Vendedor con el Auditor.
Al tener un catálogo de metadatos bien estructurado y estandarizado, éste pasa a tener un papel activo al definir quién puede ver qué información, como sucedío con el vendedor, que no tiene necesidad de ver los datos confidenciales o sensibles para hacer su trabajo.

**3.** El catálogo que construiste hoy es la base de tu PMD2. ¿Cómo crees que te servirá en las semanas 13 (calidad) y 14 (gobierno)?

En la evaluación de calidad, un pipeline puede leer el catálogo para auditar automáticamente completitud, validez y conformidad. En la parte de gobierno, al gestionar permisos de escritura, el sistema continuará consultando el nivel de sensibilidad para habilitar o restringir creación o edición de datos, columnas y tablas.

---

### Lo que debes entregar

| Entregable | Detalle |
|---|---|
| Notebook `.ipynb` | Ejecutado completo, con las 8 preguntas y la Actividad 3 respondidas |
| `farmacia_migrada.db` | Con las tablas `catalogo_metadatos` y `ficha_dublin_core` dentro |
| `catalogo_metadatos_medisur.csv` | El catálogo exportado |
| Evidencia del menú | Captura del mismo menú ejecutado como Vendedor y como Auditor |
| Portafolio | Enlace público (Colab / GitHub / Kaggle) |

### Tu avance en el PMD2

| Semana | Producto acumulativo | Estado |
|---|---|---|
| 10 | ETL y base limpia | Hecho |
| **11** | **Catálogo de metadatos + roles de lectura + menú** | **Hoy** |
| 12 | Arquitectura de datos y gráficos en el menú | Siguiente |
| 13 | Calidad de datos por dimensiones | |
| 14 | Gobierno: roles de escritura, DAMA y bitácora de auditoría | |
| 15 | Integración, KPIs y cierre del sistema | |